In [14]:
from forecasting.model_selection import forecast_metrics,cross_val
from forecasting.more_models import benchmark_forecast, model_dict
from forecasting.graphs import resid_diagnostic, future_forecast,cross_val_graph,decomp
from forecasting.bootstrap_naive_models import bs_benchmark_forecast
import pandas as pd
import plotly.graph_objects as go


class Forecast:

    def __init__(self, data, period=1):

        self.data = data
        self.period = period
        self.horizon = int(len(self.data) / 5)
        self.models = model_dict

        eval_frame = forecast_metrics(self.data,target_col='y',period=self.period)
        self.best_model = eval_frame['mean_squared_error'].idxmin()


    def best_model_forecast(self) -> pd.DataFrame:

        return benchmark_forecast(df = self.data,
                                  target_col='y',
                                  horizon=self.horizon,
                                  model=self.best_model,
                                  period=self.period,
                                  pred_width=[95,80])
    
    def run_cross_validation(self) -> dict:

        return cross_val(df = self.data,
                         target_col='y',
                         period = self.period)
    
    
    def plot_diagnostics(self, model=None) -> go.Figure:

        if model is None:
            model = self.best_model

        return resid_diagnostic(df = self.data,
                                target_col='y',
                                model = model,
                                period = self.period)
    
    def forecast(self,model=None) -> pd.DataFrame:

        if model is None:
            model = self.best_model

        return benchmark_forecast(df = self.data,
                                  target_col='y',
                                  horizon = self.horizon,
                                  model = model,
                                  pred_width=[95,80])
    
    def plot_forecast(self,model=None) -> go.Figure:

        if model is None:
            model = self.best_model

        return future_forecast(df = self.data,
                               target_col='y',
                               model = model,
                               horizon=self.horizon)
    
    def bootstrap_forecast(self,model:str) -> pd.DataFrame:

        return bs_benchmark_forecast(df = self.data,
                                     target_col='y',
                                     model = model,
                                     horizon=self.horizon)
    
    def plot_cross_validation(self,model=None) -> go.Figure:

        if model is None:
            model = self.best_model

        return cross_val_graph(df = self.data,
                               target_col='y',
                               model=model)
    
    def decomp(self) -> pd.DataFrame:

        trend, seasonal, remainder = decomp(df = self.data,
                                            target_col='y',
                                            period = self.period)
        
        components = [trend, seasonal, remainder]

        
        return pd.concat(components, axis=1)
    




    

In [15]:
import pandas as pd

df = pd.read_csv('../example_validation_data.csv', index_col='ds')
df = df[['y','forecast','fold']]
df['abs error'] = abs(df['y'] - df['forecast'])
df['error'] = df['y'] - df['forecast']

forecast = Forecast(df,7)

naive:
fold 0: 0.005998373031616211
fold 1: 0.012900590896606445
fold 2: 0.0029973983764648438
fold 3: 0.012730121612548828
fold 4: 0.0
drift:
fold 0: 0.029137372970581055
fold 1: 0.037160396575927734
fold 2: 0.0497128963470459
fold 3: 0.062445640563964844
fold 4: 0.08812236785888672
mean:
fold 0: 0.028568744659423828
fold 1: 0.05075693130493164
fold 2: 0.048821210861206055
fold 3: 2.1046183109283447
fold 4: 0.09313440322875977
ETS:
fold 0: 0.3880431652069092
fold 1: 0.7306320667266846
fold 2: 0.9676730632781982
fold 3: 1.2304551601409912
fold 4: 1.5520825386047363
ARIMA:
fold 0: 3.133474111557007
fold 1: 3.099834680557251
fold 2: 3.7652831077575684
fold 3: 12.134110450744629


16:36:49 - cmdstanpy - INFO - Chain [1] start processing


fold 4: 5.900584697723389
prophet:


16:36:50 - cmdstanpy - INFO - Chain [1] done processing
16:36:50 - cmdstanpy - INFO - Chain [1] start processing


fold 0: 0.41646313667297363


...

fold 2: 0.6831707954406738


16:36:51 - cmdstanpy - INFO - Chain [1] done processing


fold 3: 0.7183017730712891


16:36:52 - cmdstanpy - INFO - Chain [1] start processing
16:36:52 - cmdstanpy - INFO - Chain [1] done processing


fold 4: 0.8152329921722412


In [16]:
from forecasting.graphs import bootstrap_sim_graph

print(forecast.best_model)
print(forecast.best_model_forecast())
fcst = forecast.bootstrap_forecast('mean')
print(forecast.decomp())

mean
            forecast  95% lower_pi  95% upper_pi  80% lower_pi  80% upper_pi
2016-01-21  8.223865      6.528205      9.919524      7.115133      9.332597
2016-01-22  8.223865      6.528205      9.919524      7.115133      9.332597
2016-01-23  8.223865      6.528205      9.919524      7.115133      9.332597
2016-01-24  8.223865      6.528205      9.919524      7.115133      9.332597
2016-01-25  8.223865      6.528205      9.919524      7.115133      9.332597
...              ...           ...           ...           ...           ...
2017-01-14  8.223865      6.528205      9.919524      7.115133      9.332597
2017-01-15  8.223865      6.528205      9.919524      7.115133      9.332597
2017-01-16  8.223865      6.528205      9.919524      7.115133      9.332597
2017-01-17  8.223865      6.528205      9.919524      7.115133      9.332597
2017-01-18  8.223865      6.528205      9.919524      7.115133      9.332597

[364 rows x 5 columns]
               trend  seasonal     resid
ds    

In [10]:
fig = forecast.plot_diagnostics('prophet')
fig.show()

15:10:17 - cmdstanpy - INFO - Chain [1] start processing


15:10:18 - cmdstanpy - INFO - Chain [1] done processing


The 2-sided chi-squared probability for a normal hypotheis test on the residuals: 0.0000


C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

divide by zero encountered in divide

C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

invalid value encountered in multiply

